# Time Series Autoencoder for Anomaly Detection

In this exercise we build an **LSTM Autoencoder** to detect anomalies in a multivariate time series. The key idea: train the autoencoder to reconstruct *normal* data, then flag windows with high reconstruction error as anomalies.

**Dataset:** `processminer-rare-event-mts.xlsx` -- 2-minute data from a pulp-and-paper machine, 1-29 May 1999 (18,398 rows). `y = 1` marks a sheet break (124 breaks, one row each); `x1`-`x61` are process variables. Downtime after most breaks has been removed, so the time stamps have gaps. Reading `.xlsx` files needs the `openpyxl` package.

**Learning objectives:**
1. Load and explore multivariate time series data from an Excel file.
2. Split in time: train on an early normal period, set the threshold on a later normal period, evaluate on the rest.
3. Build an LSTM autoencoder (encoder + decoder) and train it on normal training windows only.
4. Use reconstruction error to raise alerts.
5. Report events found, events missed and false alarms per week, not only F1.

**Evaluation design.** Training 1-14 May, validation 15-19 May (early stopping and the threshold), test 20-29 May, scored once. The scaler and the threshold are fitted on earlier periods only. Runtime: about 1.5 minutes on an Apple M3 Max laptop CPU, including about 10 s to read the Excel file; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# Reading .xlsx files needs the openpyxl package (listed in requirements-labs.txt)
df = pd.read_excel("processminer-rare-event-mts.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
df.describe()

In [ ]:
# The label is the column "y" (1 = sheet break); the first column is the time stamp
label_col = "y"
time_col = "time"
print(f"Label column: '{label_col}'")
print(f"\nLabel distribution:")
print(df[label_col].value_counts())
print(f"\nEvent rate: {df[label_col].mean():.4%} of rows")

# Sampling interval and gaps (downtime after most breaks has been removed)
step = df[time_col].diff()
print(f"\nSampling interval: {step.mode()[0]}; longer gaps: {(step > step.mode()[0]).sum()}")

---
## 3. Visualize the Data

In [ ]:
# Sensor columns: x1-x61 without x28 (a product-grade code) and x61 (a binary flag).
# Codes are not measurements, so they are left out of the reconstruction.
feature_cols = [c for c in df.columns if c not in (label_col, time_col, "x28", "x61")]
print(f"Number of features: {len(feature_cols)}")

# Plot a few features with anomaly markers
anomaly_idx = df[df[label_col] == 1].index

fig, axes = plt.subplots(min(4, len(feature_cols)), 1, figsize=(14, 10), sharex=True)
for i, ax in enumerate(axes):
    col = feature_cols[i]
    ax.plot(df[col].values, linewidth=0.5, label=col)
    ax.scatter(anomaly_idx, df.loc[anomaly_idx, col],
               c="red", s=15, zorder=5, label="Sheet break")
    ax.set_ylabel(col, fontsize=8)
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample Index")
plt.suptitle("Feature Time Series with Anomalies Highlighted", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Split in Time and Standardize Features

Three consecutive periods: **training** (1-14 May) to fit the scaler and the autoencoder, **validation** (15-19 May) for early stopping and the threshold, and **test** (20-29 May), scored once at the end. The scaler is fitted on the normal rows of the training period only and then applied unchanged to every period.

In [ ]:
labels = df[label_col].to_numpy()
times = df[time_col]
features = df[feature_cols].to_numpy(dtype=np.float32)
assert not np.isnan(features).any()  # no missing values; otherwise impute with training statistics

# Three consecutive periods
TRAIN_END = pd.Timestamp("1999-05-15")  # training: 1-14 May
VAL_END = pd.Timestamp("1999-05-20")    # validation: 15-19 May; test: 20-29 May
period = np.select([(times < TRAIN_END).to_numpy(), (times < VAL_END).to_numpy()], [0, 1], default=2)
PERIODS = ["train", "validation", "test"]
for p, name in enumerate(PERIODS):
    rows = period == p
    print(f"{name:>10}: {times[rows].min():%d %b %H:%M} to {times[rows].max():%d %b %H:%M}, "
          f"{rows.sum():5d} rows, {labels[rows].sum():3d} breaks")

# Standardize using normal rows of the TRAINING period only
scaler = StandardScaler()
scaler.fit(features[(period == 0) & (labels == 0)])
features_scaled = scaler.transform(features)
print(f"\nFeatures shape: {features_scaled.shape}")

---
## 5. Create Sequences within Each Period

Windows are 30 rows long (60 minutes) and are labelled by their last row.

- **No window crosses a gap or a split.** A new stretch starts after every time gap (downtime) and at each period boundary, and windows are built only inside a stretch.
- **Normal windows** contain no break and do not end in the 30 minutes (15 rows) before a break or at it. The autoencoder trains on normal training windows only; normal validation windows are used for early stopping and for the threshold.
- **Per-window centring.** Several sensors drift between periods (the largest shifts are printed below, in training standard deviations). A level shift of many standard deviations would dominate the reconstruction error, and the detector would flag drift rather than unusual behaviour. We therefore subtract each window's own mean from every sensor: the autoencoder sees how the sensors move within the hour. This uses only the window itself, so it is available at every time step.

In [ ]:
SEQ_LENGTH = 30  # 30 rows = 60 minutes
LEAD_ROWS = 15   # an alert counts as a warning if it comes at most 30 minutes before a break

# Drift check on normal validation rows, in training standard deviations
val_shift = pd.Series(features_scaled[(period == 1) & (labels == 0)].mean(axis=0), index=feature_cols)
print("Largest mean shifts in the validation period (training SDs):")
print(val_shift.abs().sort_values(ascending=False).head(5).round(1).to_string())

# Contiguous stretches: a new stretch after every time gap and at every period boundary
new_stretch = (times.diff() > pd.Timedelta(minutes=2)).to_numpy() | (np.diff(period, prepend=period[0]) != 0)
stretch = np.cumsum(new_stretch)

# Window end rows: rows with at least SEQ_LENGTH - 1 earlier rows in the same stretch
ends = np.concatenate([np.flatnonzero(stretch == s)[SEQ_LENGTH - 1:] for s in np.unique(stretch)])


def create_sequences(data, ends, seq_length):
    """Windows of `seq_length` rows ending at each row in `ends`, centred per window and sensor."""
    windows = np.stack([data[e - seq_length + 1 : e + 1] for e in ends])
    return windows - windows.mean(axis=1, keepdims=True)


sequences = create_sequences(features_scaled, ends, SEQ_LENGTH)

# Window categories (each window is labelled by its last row)
event_rows = np.flatnonzero(labels == 1)
contains_break = np.array([labels[e - SEQ_LENGTH + 1 : e + 1].any() for e in ends])
warning_period = np.zeros(len(ends), dtype=bool)
for e in event_rows:
    warning_period |= (ends >= e - LEAD_ROWS) & (ends <= e)
normal = ~contains_break & ~warning_period
window_period = period[ends]

print(f"\nSequences shape: {sequences.shape}")
for p, name in enumerate(PERIODS):
    in_p = window_period == p
    print(f"{name:>10}: {in_p.sum():5d} windows, {(in_p & normal).sum():5d} normal")

In [ ]:
# Training: normal windows of the training period; validation: normal windows of the validation period
train_seq = sequences[(window_period == 0) & normal]
val_seq = sequences[(window_period == 1) & normal]

train_t = torch.tensor(train_seq, dtype=torch.float32)
val_t = torch.tensor(val_seq, dtype=torch.float32)

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(train_t, train_t),  # input == target for autoencoder
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_t, val_t),
                        batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_t)} normal windows, Val: {len(val_t)} normal windows")

---
## 6. Build LSTM Autoencoder

The autoencoder consists of:
- **Encoder**: LSTM that compresses the input sequence into a fixed-size hidden state.
- **Decoder**: LSTM that reconstructs the original sequence from the hidden state.

In [ ]:
class LSTMAutoencoder(nn.Module):
    """LSTM Autoencoder for time series anomaly detection."""

    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Encoder
        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        # Decoder
        self.decoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        # Output projection
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        batch_size, seq_len, n_features = x.shape

        # Encode: process the full input sequence
        _, (hidden, cell) = self.encoder(x)

        # Decode: reconstruct the sequence step by step
        # Start with the last input as the first decoder input
        decoder_input = x[:, -1:, :]  # (batch, 1, features)
        outputs = []

        for t in range(seq_len):
            decoder_out, (hidden, cell) = self.decoder(decoder_input, (hidden, cell))
            out = self.fc(decoder_out)  # (batch, 1, features)
            outputs.append(out)
            decoder_input = out  # use prediction as next input

        # Reverse to match original sequence order
        outputs = torch.cat(outputs, dim=1)  # (batch, seq_len, features)
        outputs = torch.flip(outputs, dims=[1])  # reverse time dimension
        return outputs


num_features = features_scaled.shape[1]
model = LSTMAutoencoder(input_size=num_features, hidden_size=64).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7. Train on Normal Data Only

The loss on normal validation windows decides when to stop. The best weights are kept as a deep copy and restored before any error is computed. If your laptop is slow, set `NUM_EPOCHS = 10`.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 20
PATIENCE = 5
train_losses, val_losses = [], []
best_val_loss, best_epoch, best_state, patience_counter = float("inf"), 0, None, 0

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    running_loss = 0.0
    for X_batch, target_batch in train_loader:
        X_batch, target_batch = X_batch.to(DEVICE), target_batch.to(DEVICE)
        optimizer.zero_grad()
        reconstructed = model(X_batch)
        loss = criterion(reconstructed, target_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    epoch_train = running_loss / len(train_t)
    train_losses.append(epoch_train)

    # Validate
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for X_batch, target_batch in val_loader:
            X_batch, target_batch = X_batch.to(DEVICE), target_batch.to(DEVICE)
            val_running += criterion(model(X_batch), target_batch).item() * X_batch.size(0)
    epoch_val = val_running / len(val_t)
    val_losses.append(epoch_val)

    # Early stopping on the validation loss
    if epoch_val < best_val_loss:
        best_val_loss, best_epoch = epoch_val, epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
              f"Train: {epoch_train:.6f}  Val: {epoch_val:.6f}")
    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)  # restore the best weights before computing errors
print(f"Restored epoch {best_epoch} with val loss: {best_val_loss:.6f}")

In [ ]:
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Chosen epoch ({best_epoch})")
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Set the Threshold on the Validation Period

The anomaly score of a window is its mean squared reconstruction error. The threshold is the 99th percentile of the errors of **normal validation windows** -- data the model never trained on -- so about 1% of normal validation windows exceed it by construction. The percentile is fixed before the test period is scored. For comparison, the cell also prints the same percentile of the training-window errors: windows the model was fitted to are not a fair reference for new data. Finally, it shows where the normal validation windows above the threshold sit: mostly within two hours after a gap in the data, when the machine restarts after downtime and start-up transients reconstruct badly. Keep this in mind when you read the test results.

In [ ]:
def reconstruction_errors(model, windows, batch_size=512):
    """Mean squared reconstruction error of each window (over time steps and sensors)."""
    model.eval()
    errors = []
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            batch = torch.tensor(windows[i : i + batch_size], dtype=torch.float32).to(DEVICE)
            errors.append(((model(batch) - batch) ** 2).mean(dim=(1, 2)).cpu().numpy())
    return np.concatenate(errors)


THRESHOLD_PERCENTILE = 99  # fixed before the test period is scored
val_errors = reconstruction_errors(model, val_seq)
threshold = np.percentile(val_errors, THRESHOLD_PERCENTILE)
print(f"Threshold ({THRESHOLD_PERCENTILE}th percentile of normal validation errors): {threshold:.4f}")

print(f"Median of normal validation errors: {np.median(val_errors):.4f}")

train_errors = reconstruction_errors(model, train_seq)
print(f"Same percentile of training-window errors, for comparison: "
      f"{np.percentile(train_errors, THRESHOLD_PERCENTILE):.4f}")

# Where do the normal validation windows above the threshold sit? Minutes since the last gap
restart_rows = np.flatnonzero(times.diff().to_numpy() > np.timedelta64(2, "m"))  # first row after each gap
val_ends = ends[(window_period == 1) & normal]
last_restart = restart_rows[np.searchsorted(restart_rows, val_ends, side="right") - 1]
minutes_since_restart = 2 * (val_ends - last_restart)
above = val_errors > threshold
print(f"Normal validation windows above the threshold: {above.sum()}; "
      f"{np.mean(minutes_since_restart[above] <= 120):.0%} of them within 2 hours after a gap "
      f"(all normal validation windows: {np.mean(minutes_since_restart <= 120):.0%})")

---
## 9. Score the Test Period Once: Events, Misses and False Alarms

Operators do not see windows; they see **alerts**. An alert starts at the first window above the threshold after a quiet one and lasts while consecutive windows stay above it. We count:
- **events found**: breaks with an alert raised at most 30 minutes before the break, or at it; the lead time runs from the first such alert to the break;
- **events missed**: all other breaks (a few come too soon after a restart for a complete 60-minute window);
- **false alarms**: alerts that are neither such a warning nor raised in the hour after a break (while the window still contains it), also as a rate per week of operation.

The test period is scored once, with the threshold and all settings fixed above.

In [ ]:
test_mask = window_period == 2
test_ends = ends[test_mask]
test_errors = reconstruction_errors(model, sequences[test_mask])
flagged = test_errors > threshold

# Alerts: the first window of each run of consecutive flagged windows within one stretch
continues = np.r_[False, flagged[:-1] & (np.diff(test_ends) == 1)
                  & (np.diff(stretch[test_ends]) == 0)]
alert_rows = test_ends[flagged & ~continues]

test_events = event_rows[period[event_rows] == 2]
is_warning = np.zeros(len(alert_rows), dtype=bool)
is_late = np.zeros(len(alert_rows), dtype=bool)
lead_minutes = []
for e in test_events:
    warns = (alert_rows >= e - LEAD_ROWS) & (alert_rows <= e)
    is_warning |= warns
    is_late |= (alert_rows > e) & (alert_rows - SEQ_LENGTH + 1 <= e)  # window still contains the break
    if warns.any():
        lead_minutes.append(2 * (e - alert_rows[warns].min()))
false_alarms = int((~is_warning & ~is_late).sum())

weeks = test_mask.sum() * 2 / (60 * 24 * 7)  # operating time covered by the test windows
detectable = sum(((test_ends >= e - LEAD_ROWS) & (test_ends <= e)).any() for e in test_events)
median_lead = f"{np.median(lead_minutes):.0f} min" if lead_minutes else "n/a"
print(f"Test period: {len(test_events)} breaks ({detectable} with a complete window before them), "
      f"{weeks:.2f} weeks of operation")
print(f"Events found:  {len(lead_minutes)}   ({sum(m > 0 for m in lead_minutes)} before the break itself; "
      f"median lead time {median_lead})")
print(f"Events missed: {len(test_events) - len(lead_minutes)}")
print(f"False alarms:  {false_alarms}   ({false_alarms / weeks:.0f} per week); "
      f"alerts in the hour after a break: {int(is_late.sum())}")
print(f"Share of test windows above the threshold: {flagged.mean():.1%}")
print(f"Share of alerts that were warnings: {is_warning.mean():.0%} "
      f"(for a randomly timed alert: {warning_period[test_mask].mean():.0%})")

In [ ]:
# For contrast: window-level precision, recall and F1, a common but misleading way to
# score event detection (a window is positive when its last row is a break)
window_is_break = labels[test_ends] == 1
tp = int((flagged & window_is_break).sum())
fp = int((flagged & ~window_is_break).sum())
fn = int((~flagged & window_is_break).sum())

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Windows: TP {tp}, FP {fp}, FN {fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("A window-level F1 mixes events, lead time and false alarms; report the event counts above.")

---
## 10. Visualize Detected Anomalies

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Reconstruction error over the test period, with the threshold and the breaks
test_times = times.to_numpy()[test_ends]
axes[0].plot(test_times, test_errors, linewidth=0.5, label="Reconstruction Error")
axes[0].axhline(y=threshold, color="r", linestyle="--", label=f"Threshold from validation ({threshold:.3f})")
for i, t in enumerate(times.to_numpy()[test_events]):
    axes[0].axvline(t, color="red", alpha=0.25, linewidth=0.8, label="Sheet break" if i == 0 else None)
axes[0].set_yscale("log")
axes[0].set_title("Reconstruction Error Over the Test Period")
axes[0].set_ylabel("MSE (log scale)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Distribution of test errors: normal windows vs windows in the 30 minutes before a break
bins = np.logspace(np.log10(test_errors.min()), np.log10(test_errors.max()), 60)
axes[1].hist(test_errors[normal[test_mask]], bins=bins, alpha=0.6, label="Normal test windows", density=True)
axes[1].hist(test_errors[warning_period[test_mask]], bins=bins, alpha=0.6,
             label="Test windows in the 30 min before a break", density=True)
axes[1].axvline(x=threshold, color="r", linestyle="--", label="Threshold (from validation)")
axes[1].set_xscale("log")
axes[1].set_title("Distribution of Reconstruction Errors in the Test Period")
axes[1].set_xlabel("MSE (log scale)")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Summary

In this notebook we:

1. Loaded a multivariate process time series with rare sheet breaks (`y = 1`) from an Excel file.
2. Split it in time: training 1-14 May, validation 15-19 May, test 20-29 May.
3. Standardized the sensors with statistics of normal training rows only.
4. Built 60-minute windows inside contiguous stretches (no window crosses a gap or a split) and centred each window to remove slow drift.
5. Trained an LSTM autoencoder on normal training windows, with early stopping on normal validation windows.
6. Set the threshold at the 99th percentile of normal validation errors.
7. Scored the test period once: events found, events missed and false alarms per week, with a window-level F1 only for contrast.

**Key takeaways:**
- Autoencoders learn to reconstruct normal patterns. Anomalous data produces higher reconstruction error.
- Training on normal data only is critical -- including anomalies would teach the model to reconstruct them too.
- Fit the scaler on the training period and the threshold on a later normal period; errors on training windows are not a fair reference.
- The threshold trades missed events for false alarms; choose it from an alarm budget operators can live with.
- Caveat for this run: start-up transients after restarts set the 99th percentile of the normal validation errors, far above the bulk of normal errors, and most test breaks stay below it. A lower threshold would catch more breaks but also flag more of the test period, whose typical errors are higher than in validation (drift again). Excluding the first hours after a restart, or an alarm-budget threshold, would change the trade-off -- judge such changes on validation or a new period, not on this test set.
- Count events, lead times and false alarms per week: overlapping windows repeat one event many times, and a window-level F1 hides all three.
- Sensor levels drift between periods; without per-window centring a new regime looks anomalous. Retrain and re-threshold on a schedule.
- This approach is *unsupervised* in the sense that we do not need labelled anomalies for training, only for evaluation.